# Diffusion of a Gaussian function
Time-dependent problem: <br>
$\frac{\partial u}{\partial t} = \nabla^2 u + f$ in $\Omega \times (0,T]$ <br>
$u=u_D$ on $\partial \Omega \times (0,T]$ <br>
$u=u_0$ at $t=0$ <br>
Discretization in time by backward finite differences. <br>
Here: $u_0(x,y)=\exp(-a x^2-ay^2)$ for $a=5$ on the domain $[-2,2]\times[-2,2]$, with homogeneous Dirichlet boundary conditions $u_D=0$.

In [1]:
import matplotlib as mpl
import pyvista
import ufl
import numpy as np

from petsc4py import PETSc
from mpi4py import MPI

from dolfinx import fem, mesh, io, plot
from dolfinx.fem.petsc import (
    assemble_vector,
    assemble_matrix,
    create_vector,
    apply_lifting,
    set_bc,
)

In [2]:
# Define time discretization parameters
t = 0.0 # start time
T = 1.0 # final time
num_steps = 50
dt = T/num_steps # time step size

In [3]:
# Define mesh
nx, ny = 50, 50
domain = mesh.create_rectangle(MPI.COMM_WORLD, 
                               [np.array([-2,-2]), np.array([2, 2])],
                               [nx, ny],
                               mesh.CellType.triangle)
V = fem.functionspace(domain, ("Lagrange", 1))

In [4]:
# Initial condition
def initial_condition(x, a=5):
    return np.exp(-a* (x[0]**2 + x[1]**2))

u_n = fem.Function(V)
u_n.name = "u_n"
u_n.interpolate(initial_condition)

# Boundary condition
fdim = domain.topology.dim - 1
boundary_facets = mesh.locate_entities_boundary(domain, fdim, lambda x: np.full(x.shape[1], True, dtype=bool))
bc = fem.dirichletbc(PETSc.ScalarType(0), fem.locate_dofs_topological(V, fdim, boundary_facets), V)

In [5]:
uh = fem.Function(V)
uh.name = "uh"
uh.interpolate(initial_condition)

In [6]:
# Define variational problem
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
f = fem.Constant(domain, PETSc.ScalarType(0))
a = u*v*ufl.dx + dt*ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = (u_n + dt*f)*v*ufl.dx

In [7]:
# Generate assembly kernels for the matrix and vector
bilinear_form = fem.form(a)
linear_form = fem.form(L)

# Assemble matrix (not time dependent)
A = assemble_matrix(bilinear_form, bcs=[bc])
A.assemble()
b = create_vector(fem.extract_function_spaces(linear_form))# only create vector, don't assemble, as it changes in every time step

In [8]:
# Create krylov subspace solver (cannot use LinearProblem anymore because matrix is already assembled)
solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
solver.getPC().setType(PETSc.PC.Type.LU)

In [9]:
# Visualization of time dependent problem using pyvista
pyvista.set_jupyter_backend("html")
grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(V))

plotter = pyvista.Plotter()
plotter.open_gif("u_time.gif", fps=10)

grid.point_data["uh"] = uh.x.array
warped = grid.warp_by_scalar("uh", factor=1)

viridis = mpl.colormaps.get_cmap("viridis").resampled(25)
sargs = dict(
    title_font_size=25,
    label_font_size=20,
    fmt="%.2e",
    color="black",
    position_x=0.1,
    position_y=0.8,
    width=0.8,
    height=0.1,
)

renderer = plotter.add_mesh(
    warped,
    show_edges=True,
    lighting=False,
    cmap=viridis,
    scalar_bar_args=sargs,
    clim=[0, max(uh.x.array)],
)

2026-03-06 08:55:29.516 (   1.026s) [    707CA7E24600]vtkXOpenGLRenderWindow.:1458  WARN| bad X server connection. DISPLAY=


### Updating the solution and right hand side per time step

To be able to solve the variation problem at each time step, we have to assemble the right hand side and apply the boundary condition before calling `solver.solve(b, uh.x.petsc_vec)`. We start by resetting the values in `b` as we are reusing the vector at every time step. The next step is to assemble the vector calling `dolfinx.fem.petsc.assemble_vector(b, L)`, which means that we are assembling the linear form `L(v)` into the vector `b`. Note that we do not supply the boundary conditions for assembly, as opposed to the left hand side. This is because we want to use lifting to apply the boundary condition, which preserves symmetry of the matrix $A$ in the bilinear form $a(u,v)=a(v,u)$ without Dirichlet boundary conditions. Once we have performed the lifting, we accumulate values from degrees of freedom that are shared between processes using `b.ghostUpdate()`. Finally, we apply the boundary condition to the fixed degrees of freedom with `dolfinx.fem.petsc.set_bc()`. Next, we can solve the linear system and update degrees of freedom shared between processors. Finally, before moving to the next time step, we update the solution at the previous time step to the solution at this time step.

In [10]:
for i in range(num_steps):
    t += dt

    # Update the right hand side reusing the initial vector
    with b.localForm() as loc_b:
        loc_b.set(0)
    assemble_vector(b, linear_form)

    # Apply Dirichlet boundary condition to the vector
    apply_lifting(b, [bilinear_form], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # Solve linear problem
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # Update solution at previous time step (u_n)
    u_n.x.array[:] = uh.x.array
   
    # Update plot
    new_warped = grid.warp_by_scalar("uh", factor=1)
    warped.points[:, :] = new_warped.points
    warped.point_data["uh"][:] = uh.x.array
    plotter.write_frame()
plotter.close()

# Destroy PETSc objects to avoid memory leaks.
A.destroy()
b.destroy()
solver.destroy()